# SD runtime cold start probe

该 Notebook 用于 Colab 冷启动: 拉取仓库代码, 安装依赖, 登录 Hugging Face, 加载真实 SD3.5 Medium 主模型, 可选运行 SD3 Medium 兼容对照, 捕获真实 latent trajectory, 并把 outputs 下的运行产物打包为 zip 保存到 Google Drive 的 SLM 目录。

正式逻辑位于 `paper_workflow/colab_utils/sd_runtime_cold_start.py`, Notebook 只作为远程执行入口。


## 运行前准备

1. 在 Colab 中选择 GPU runtime。
2. 准备 Hugging Face token, 并确认账号已获得目标模型访问权限。
3. 确认 Google Drive 可挂载。默认保存目录为 `/content/drive/MyDrive/SLM/real_sd_runtime_probe`。
4. 默认 `SLM_WM_MODEL_SELECTION=both`: 同时运行 SD3.5 Medium 主模型和 SD3 Medium 兼容对照。若只想先跑主模型, 设置为 `auto`。


In [ ]:
%pip install -q --upgrade diffusers transformers accelerate safetensors sentencepiece protobuf huggingface_hub


In [ ]:
import os
import subprocess
import sys

repository_url = os.environ.get('SLM_WM_REPOSITORY_URL', 'https://github.com/RICHAAARC/SLM-WM.git')
repository_ref = os.environ.get('SLM_WM_REPOSITORY_REF', 'main')
workspace_dir = os.environ.get('SLM_WM_WORKSPACE_DIR', '/content/slm_wm_repository')

if not os.path.exists(workspace_dir):
    subprocess.run(['git', 'clone', repository_url, workspace_dir], check=True)

subprocess.run(['git', 'fetch', '--all'], cwd=workspace_dir, check=True)
subprocess.run(['git', 'checkout', repository_ref], cwd=workspace_dir, check=True)
os.chdir(workspace_dir)
if workspace_dir not in sys.path:
    sys.path.insert(0, workspace_dir)
print({'workspace_dir': workspace_dir, 'repository_ref': repository_ref})


In [ ]:
import os

try:
    from google.colab import userdata
    token_from_secret = userdata.get('HF_TOKEN')
except Exception:
    token_from_secret = None

if token_from_secret and not os.environ.get('HF_TOKEN'):
    os.environ['HF_TOKEN'] = token_from_secret

if os.environ.get('HF_TOKEN'):
    from huggingface_hub import login
    login(token=os.environ['HF_TOKEN'])
    print('huggingface_login_ready')
else:
    print('HF_TOKEN_not_found')


In [ ]:
import torch

device_report = {
    'cuda_available': torch.cuda.is_available(),
    'device_count': torch.cuda.device_count(),
    'device_name': torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none',
}
print(device_report)
assert device_report['cuda_available'], '需要 Colab GPU runtime'


In [ ]:
import os

from paper_workflow.colab_utils.sd_runtime_cold_start import run_default_model_plan

model_selection = os.environ.get('SLM_WM_MODEL_SELECTION', 'both')
runtime_summaries = run_default_model_plan(root='.', model_selection=model_selection)
runtime_summaries


In [ ]:
import os

from google.colab import drive

from paper_workflow.colab_utils.sd_runtime_cold_start import package_probe_outputs

drive.mount('/content/drive')
drive_output_dir = os.environ.get('SLM_WM_DRIVE_OUTPUT_DIR', '/content/drive/MyDrive/SLM/real_sd_runtime_probe')
archive_record = package_probe_outputs(root='.', drive_output_dir=drive_output_dir)
archive_record.to_dict()


In [ ]:
from pathlib import Path

for summary_path in sorted(Path('outputs/real_sd_runtime_probe').glob('*_summary.json')):
    print(summary_path)
    print(summary_path.read_text(encoding='utf-8'))
